假設有一筆資料 data = { 'size': ['S','M',np.nan,'XL','XL'], 'color': ['red', 'blue', 'blue', 'black', np.nan], 'price': [2100, np.nan, 4500, 7300, 3200], 'quantity': [np.nan, 350, np.nan, 200, 10] } df = pd.DataFrame(data)

In [17]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

data = {'size': ['S', 'M', np.nan, 'XL', 'XL'],
        'color': ['red', 'blue', 'blue', 'black', np.nan],
        'price': [2100, np.nan, 4500, 7300, 3200],
        'quantity': [np.nan, 350, np.nan, 200, 10]}
df = pd.DataFrame(data)
print('原始資料')
df


原始資料


,size,color,price,quantity
0,S,red,2100.0,NaN
1,M,blue,NaN,350.0
2,NaN,blue,4500.0,NaN
3,XL,black,7300.0,200.0
4,XL,NaN,3200.0,10.0


(1) 請用ColumnTransformer水平合併器，將數值和類別管道器結合起來。其中數值管道器要做遺漏值處理，用中位數median，再做MinMaxScaler轉換。類別管道器的遺漏值用眾數處理後，再做獨熱編碼。

In [19]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder())])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, ['price', 'quantity']),
        ('cat', categorical_transformer, ['size', 'color'])])

transformed_data = preprocessor.fit_transform(df)
print('轉換後資料')
transformed_data

轉換後資料


array([[0.        , 0.55882353, 0.        , 1.        , 0.        ,
        0.        , 0.        , 1.        ],
       [0.33653846, 1.        , 1.        , 0.        , 0.        ,
        0.        , 1.        , 0.        ],
       [0.46153846, 0.55882353, 0.        , 0.        , 1.        ,
        0.        , 1.        , 0.        ],
       [1.        , 0.55882353, 0.        , 0.        , 1.        ,
        1.        , 0.        , 0.        ],
       [0.21153846, 0.        , 0.        , 0.        , 1.        ,
        0.        , 1.        , 0.        ]])

(2) 請將獨熱編碼的欄位和數值型資料欄位取出，並整合到DataFrame裡

In [18]:
onehot_feature_names = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(input_features=['size', 'color'])
encoded_features = pd.DataFrame(transformed_data, columns=np.concatenate([['price', 'quantity'], onehot_feature_names]))
print('結合後資料')
encoded_features

結合後資料


,price,quantity,size_M,size_S,size_XL,color_black,color_blue,color_red
0,0.000000,0.558824,0.0,1.0,0.0,0.0,0.0,1.0
1,0.336538,1.000000,1.0,0.0,0.0,0.0,1.0,0.0
2,0.461538,0.558824,0.0,0.0,1.0,0.0,1.0,0.0
3,1.000000,0.558824,0.0,0.0,1.0,1.0,0.0,0.0
4,0.211538,0.000000,0.0,0.0,1.0,0.0,1.0,0.0
